In [ ]:
import ee

# Esto abrirá una ventana en tu navegador para que inicies sesión 
# con la cuenta de Google que tiene acceso a Earth Engine.
ee.Authenticate()

# Una vez autenticado, inicializamos la API
ee.Initialize()

print("¡Google Earth Engine se ha inicializado correctamente!")

## 4/1AXEQxIBdgVJ8vMqHHBLBO0bepfCO9ngmrTr7TkDfx25Kra4WSwE1ASeYJWg


Successfully saved authorization token.
¡Google Earth Engine se ha inicializado correctamente!


In [41]:
# Un pequeño test para verificar que podemos consultar datos
# Vamos a pedirle a GEE que nos imprima la elevación promedio de un punto en el mundo
test_point = ee.Geometry.Point([145.3725, -16.4632]) # El punto que definimos antes
dem = ee.Image('USGS/SRTMGL1_003')

elevation = dem.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=test_point,
    scale=30
).get('elevation')

print(f"Elevación del punto de prueba: {elevation.getInfo()} metros")

Elevación del punto de prueba: 16 metros


In [4]:
import requests

def get_soil_data(lat, lon):
    """
    Consulta la API de SoilGrids para obtener propiedades del suelo en un punto.
    """
    # URL de la API de ISRIC (SoilGrids)
    url = f"https://rest.isric.org/soilgrids/v2.0/properties/query?lon={lon}&lat={lat}&property=phh2o&property=soc&property=clay&property=sand&property=silt&property=bdod&depth=0-30cm&value=mean"
    
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        # Aquí procesaríamos el JSON para extraer solo los valores numéricos
        return data
    else:
        return f"Error en la consulta: {response.status_code}"

# Ejemplo de prueba para El Playón
data = get_soil_data(7.35, -73.23)
print(data)

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-73.23, 7.35]}, 'properties': {'layers': []}, 'query_time_s': 0.009196996688842773}


In [5]:
import requests

def get_soil_data_v2(lat, lon):
    # La API de SoilGrids v2.0 requiere que las propiedades se separen con comas
    # y los valores sean explícitos.
    base_url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    
    # Construimos los parámetros como cadenas de texto simples
    params = {
        'lon': lon,
        'lat': lat,
        'property': 'phh2o,soc,clay,sand,silt,bdod',
        'depth': '0-30cm',
        'value': 'mean'
    }
    
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error {response.status_code}: {response.text}"

# Prueba de nuevo
data = get_soil_data_v2(-73.1867,7.3297)
print(data)

Error 500: Internal Server Error


In [46]:
import ee

# 1. Definimos el punto
lat, lon = 7.3297, -73.1867
point = ee.Geometry.Point([lon, lat])

# 2. Cargamos la imagen de pH de SoilGrids
# Usaremos la colección genérica de ISRIC para estar seguros
soil_image = ee.Image("projects/soilgrids-isric/phh2o_mean")

# 3. ¡EL TRUCO! Imprimimos los nombres de las bandas disponibles
print("Bandas disponibles en la imagen:", soil_image.bandNames().getInfo())

# 4. Extraemos el valor usando el nombre que nos arroje el print de arriba
stats = soil_image.reduceRegion(
    reducer=ee.Reducer.first(), # Usamos 'first' para obtener el valor del píxel exacto
    geometry=point,
    scale=250
)

print("Resultado de la extracción:", stats.getInfo())

Bandas disponibles en la imagen: ['phh2o_0-5cm_mean', 'phh2o_5-15cm_mean', 'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_60-100cm_mean', 'phh2o_100-200cm_mean']
Resultado de la extracción: {'phh2o_0-5cm_mean': 52, 'phh2o_100-200cm_mean': 54, 'phh2o_15-30cm_mean': 53, 'phh2o_30-60cm_mean': 53, 'phh2o_5-15cm_mean': 53, 'phh2o_60-100cm_mean': 54}


In [7]:
import ee

def get_soil_data_gee(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    
    # Abrimos la imagen de SoilGrids
    soil_image = ee.Image("projects/soilgrids-isric/phh2o_mean")
    
    # Esto nos ayudará a depurar si necesitamos otra banda:
    # print("Bandas disponibles:", soil_image.bandNames().getInfo())
    
    # Extraer el valor en el punto
    # Nota: usamos 'phh2o_mean' que es el nombre estándar en GEE para esta capa
    stats = soil_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=250,
        bestEffort=True # Por si el punto cae al borde
    )
    
    # Obtenemos el valor usando el nombre correcto de la banda
    value = stats.get('phh2o_30-60cm_mean')
    
    return value.getInfo()

# Probemos ahora
ph_value = get_soil_data_gee(7.3297, -73.1867)
print(f"Valor de pH (EE): {ph_value}")

Valor de pH (EE): 53


cec (Cation Exchange Capacity): Capacidad de intercambio catiónico (indica la fertilidad potencial).

ocd (Organic Carbon Density): Densidad de carbono orgánico.

cfvo (Coarse Fragments): Fragmentos gruesos (pedregosidad del suelo).

soc (Organic Carbon): Ya lo tienes como soc (Soil Organic Carbon).

bdod Bulk Density of the fine earth fraction (Densidad aparente de la fracción de tierra fina)

In [8]:
import ee

def get_soil_profile(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    
    # Mapeamos las propiedades que necesitamos a las colecciones de GEE
    # Nota: SoilGrids en GEE usa estos IDs de proyecto
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    results = {}
    
    for prop, collection_id in properties.items():
        try:
            image = ee.Image(collection_id)
            # Extraemos la capa superficial (0-5cm) como referencia inicial
            band_name = f"{prop}_0-5cm_mean"
            
            val = image.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=250,
                bestEffort=True
            ).get(band_name)
            
            results[prop] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probemos la extracción masiva
soil_data = get_soil_profile(7.3297, -73.1867)
print("Perfil de suelo extraído:", soil_data)

Perfil de suelo extraído: {'phh2o': 52, 'soc': 561, 'clay': 346, 'sand': 311, 'silt': 344, 'bdod': 99, 'cec': 179}


In [9]:
import ee

# Nombre de la colección base que estamos usando
collection_id = "projects/soilgrids-isric/phh2o_mean" 

# Inspeccionamos los metadatos de la imagen
image = ee.Image(collection_id)
print("Propiedades de la imagen (Metadatos):", image.propertyNames().getInfo())
print(len(image.propertyNames().getInfo()))

Propiedades de la imagen (Metadatos): ['Covariates', 'WoSIS_version', 'description', 'system:id', 'Mtry', 'Litter_layers', 'Mapped_units', 'title', 'Number_trees', 'Outputs_version', 'system:footprint', 'Model', 'system:version', 'Code_version', 'Model_type', 'system:asset_size', 'system:bands', 'system:band_names']
18


In [10]:
import ee

def get_historical_precip(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    
    # Cargar colección CHIRPS
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    # Calcular el promedio de toda la serie histórica disponible (1981-presente)
    # y extraer el valor en tu punto
    precip_mean = chirps.mean().reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=5000, # CHIRPS tiene resolución de ~5km
        bestEffort=True
    ).get('precipitation')
    
    return precip_mean.getInfo()

# Ejecutar para El Playón
precip_val = get_historical_precip(7.3297, -73.1867)
print(f"Precipitación histórica promedio (pentad): {precip_val} mm")

Precipitación histórica promedio (pentad): 22.629444122314453 mm


In [11]:
import ee

def get_yearly_precip_last_5_years(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    # Definimos el rango: últimos 5 años completos (2020 a 2026)
    start_year = 2020
    end_year = 2026
    
    yearly_data = []
    
    for year in range(start_year, end_year + 1):
        # Filtramos la colección por el año específico
        yearly_image = chirps.filter(ee.Filter.calendarRange(year, year, 'year')).mean()
        
        # Extraemos el valor del punto
        value = yearly_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=5000,
            bestEffort=True
        ).get('precipitation')
        
        # Guardamos el resultado en una lista
        val_info = value.getInfo()
        yearly_data.append({'year': year, 'precipitation': val_info})
        
    return yearly_data

# Prueba
precip_5_years = get_yearly_precip_last_5_years(7.3297, -73.1867)
print("Precipitación anual (últimos 5 años):", precip_5_years)

Precipitación anual (últimos 5 años): [{'year': 2020, 'precipitation': 20.709693908691406}, {'year': 2021, 'precipitation': 22.06425666809082}, {'year': 2022, 'precipitation': 27.816709518432617}, {'year': 2023, 'precipitation': 23.727134704589844}, {'year': 2024, 'precipitation': 24.230972290039062}, {'year': 2025, 'precipitation': 23.866689682006836}, {'year': 2026, 'precipitation': 27.423357009887695}]


In [12]:
import ee

def get_seasonal_precip(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    # Definimos los trimestres (meses)
    quarters = {
        "Q1_Jan_Mar": [1, 3],
        "Q2_Apr_Jun": [4, 6],
        "Q3_Jul_Sep": [7, 9],
        "Q4_Oct_Dec": [10, 12]
    }
    
    seasonal_data = {}
    
    for name, months in quarters.items():
        # Promedio histórico para esos meses (1981-2025)
        seasonal_avg = chirps.filter(ee.Filter.calendarRange(months[0], months[1], 'month')) \
                             .filter(ee.Filter.calendarRange(1981, 2025, 'year')) \
                             .mean()
        
        # Extraer valor en el punto
        val = seasonal_avg.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=5000,
            bestEffort=True
        ).get('precipitation')
        
        seasonal_data[name] = val.getInfo()
        
    return seasonal_data

# Prueba
print("Promedio histórico trimestral (mm):", get_seasonal_precip(7.3297, -73.1867))

Promedio histórico trimestral (mm): {'Q1_Jan_Mar': 17.405012130737305, 'Q2_Apr_Jun': 25.39470863342285, 'Q3_Jul_Sep': 18.19121742248535, 'Q4_Oct_Dec': 29.31377410888672}


In [13]:
import ee

def get_precip_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    quarters = {
        "Q1": [1, 3], "Q2": [4, 6], "Q3": [7, 9], "Q4": [10, 12]
    }
    
    report = {}
    
    for year in range(start_year, end_year + 1):
        report[year] = {}
        for q_name, months in quarters.items():
            # Filtramos la colección
            filtered = chirps.filter(ee.Filter.calendarRange(year, year, 'year')) \
                             .filter(ee.Filter.calendarRange(months[0], months[1], 'month'))
            
            # Verificamos si la colección tiene imágenes antes de procesar
            count = filtered.size().getInfo()
            
            if count > 0:
                period_image = filtered.mean()
                
                # Obtenemos estadísticos
                stats = period_image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=point,
                    scale=5000,
                    bestEffort=True
                )
                
                # Obtenemos las claves disponibles
                keys = stats.keys().getInfo()
                
                if len(keys) > 0:
                    val = stats.get(keys[0]).getInfo()
                    report[year][q_name] = val
                else:
                    report[year][q_name] = 0
            else:
                # Si no hay imágenes, el valor es 0
                report[year][q_name] = 0
            
    return report

# Ejecución
start = 2020
end = 2026
data_matrix = get_precip_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(data_matrix, indent=4))

{
    "2020": {
        "Q1": 18.679712295532227,
        "Q2": 18.388031005859375,
        "Q3": 16.53814697265625,
        "Q4": 29.232887268066406
    },
    "2021": {
        "Q1": 12.420306205749512,
        "Q2": 24.085519790649414,
        "Q3": 20.859636306762695,
        "Q4": 30.891565322875977
    },
    "2022": {
        "Q1": 21.37236213684082,
        "Q2": 29.619091033935547,
        "Q3": 20.573762893676758,
        "Q4": 39.70161819458008
    },
    "2023": {
        "Q1": 22.015317916870117,
        "Q2": 30.92342185974121,
        "Q3": 16.014013290405273,
        "Q4": 25.955787658691406
    },
    "2024": {
        "Q1": 13.816878318786621,
        "Q2": 28.686704635620117,
        "Q3": 17.992687225341797,
        "Q4": 36.427616119384766
    },
    "2025": {
        "Q1": 18.3424015045166,
        "Q2": 23.997785568237305,
        "Q3": 24.884586334228516,
        "Q4": 28.241989135742188
    },
    "2026": {
        "Q1": 31.3807315826416,
        "Q2": 23.46598

In [14]:
import ee

def get_precip_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    quarters = {
        "Q1": [1, 3], "Q2": [4, 6], "Q3": [7, 9], "Q4": [10, 12]
    }
    
    report = {}
    
    for year in range(start_year, end_year + 1):
        report[year] = {}
        for q_name, months in quarters.items():
            # Filtramos la colección
            filtered = chirps.filter(ee.Filter.calendarRange(year, year, 'year')) \
                             .filter(ee.Filter.calendarRange(months[0], months[1], 'month'))
            
            # Verificamos si la colección tiene imágenes antes de procesar
            count = filtered.size().getInfo()
            
            if count > 0:
                period_image = filtered.mean()
                
                # Obtenemos estadísticos
                stats = period_image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=point,
                    scale=5000,
                    bestEffort=True
                )
                
                # Obtenemos las claves disponibles
                keys = stats.keys().getInfo()
                
                if len(keys) > 0:
                    val = stats.get(keys[0]).getInfo()
                    report[year][q_name] = val
                else:
                    report[year][q_name] = 0
            else:
                # Si no hay imágenes, el valor es 0
                report[year][q_name] = 0
            
    return report

# Ejecución
start = 2024
end = 2026
data_matrix = get_precip_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(data_matrix, indent=4))

{
    "2024": {
        "Q1": 13.816878318786621,
        "Q2": 28.686704635620117,
        "Q3": 17.992687225341797,
        "Q4": 36.427616119384766
    },
    "2025": {
        "Q1": 18.3424015045166,
        "Q2": 23.997785568237305,
        "Q3": 24.884586334228516,
        "Q4": 28.241989135742188
    },
    "2026": {
        "Q1": 31.3807315826416,
        "Q2": 23.46598243713379,
        "Q3": 0,
        "Q4": 0
    }
}


In [15]:
import ee
import math

def get_topography_area_robust(lat, lon, area_meters=56):
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    dem = ee.Image("USGS/SRTMGL1_003")
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem)
    
    # Calcular componentes vectoriales y renombrarlos explícitamente
    aspect_rad = aspect.multiply(math.pi / 180)
    sin_aspect = aspect_rad.sin().rename('sin')
    cos_aspect = aspect_rad.cos().rename('cos')
    
    # Combinamos renombrando el DEM también
    combined = ee.Image.cat([dem.rename('elevation'), slope, sin_aspect, cos_aspect])
    
    stats = combined.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=30,
        bestEffort=True
    ).getInfo()
    
    # Reconvertimos los componentes vectoriales a grados
    final_aspect = (math.atan2(stats['sin'], stats['cos']) * 180 / math.pi) % 360
    
    return {
        'elevation': stats['elevation'],
        'slope': stats['slope'],
        'aspect': final_aspect
    }

# Prueba la versión corregida
topo_robust = get_topography_area_robust(7.3297, -73.1867)
print(topo_robust)

{'elevation': 1060.3586804584847, 'slope': 17.176286736712285, 'aspect': 195.3507764722462}


El resultado null (o None en Python) en Earth Engine suele suceder por una de dos razones principales: la resolución espacial o la geometría de la región.

En tu código anterior, usamos scale=5000 (la resolución de CHIRPS). Al usar buffer(56).bounds(), le pedimos a Earth Engine que calcule el promedio dentro de un área de 1 hectárea (que es muy pequeña, aproximadamente 112m x 112m).
¿Por qué está dando null?

    El problema de la Escala: Si tu región de interés (1 hectárea) es más pequeña que un solo píxel de CHIRPS (que mide 5,000 metros), Earth Engine a veces tiene dificultades para determinar qué píxel tomar o cómo hacer el remuestreo si el punto central no cae perfectamente dentro de una celda de datos.

    El "Bounds" excesivo: Al usar .bounds(), creamos un cuadrado. Si este cuadrado es diminuto, Earth Engine intenta buscar datos dentro de él y, si no hay un píxel que cubra completamente el área o si el remuestreo falla, devuelve null.

In [ ]:
import ee
import json

def get_precip_by_year_and_quarter_area(lat, lon, start_year, end_year, area_meters=100):
    # Definimos el área de interés (1 hectárea aprox = 56m de radio)
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
    
    quarters = {
        "Q1": [1, 3], "Q2": [4, 6], "Q3": [7, 9], "Q4": [10, 12]
    }
    
    report = {}
    
    for year in range(start_year, end_year + 1):
        report[year] = {}
        for q_name, months in quarters.items():
            # Filtramos
            filtered = chirps.filter(ee.Filter.calendarRange(year, year, 'year')) \
                             .filter(ee.Filter.calendarRange(months[0], months[1], 'month'))
            
            # Verificamos si hay imágenes
            count = filtered.size().getInfo()
            
            if count > 0:
                # Calculamos el promedio espacial dentro de la región y temporal del trimestre
                period_image = filtered.mean()
                
                stats = period_image.reduceRegion(
                    reducer=ee.Reducer.mean(), # Promedia los píxeles dentro de la hectárea
                    geometry=region,
                    scale=100, # CHIRPS es 0.1km, esto está bien para una zona
                    bestEffort=True
                )
                
                keys = stats.keys().getInfo()
                if len(keys) > 0:
                    val = stats.get(keys[0]).getInfo()
                    report[year][q_name] = val
                else:
                    report[year][q_name] = 0
            else:
                report[year][q_name] = 0
            
    return report

# Ejecución para el área de 1 hectárea
start = 2024
end = 2026
data_matrix = get_precip_by_year_and_quarter_area(7.3297, -73.1867, start, end)

print(json.dumps(data_matrix, indent=4))

{
    "2024": {
        "Q1": 13.816878318786621,
        "Q2": 28.686704635620117,
        "Q3": 17.992687225341797,
        "Q4": 36.427616119384766
    },
    "2025": {
        "Q1": 18.3424015045166,
        "Q2": 23.997785568237305,
        "Q3": 24.884586334228516,
        "Q4": 28.241989135742188
    },
    "2026": {
        "Q1": 31.3807315826416,
        "Q2": 23.46598243713379,
        "Q3": 0,
        "Q4": 0
    }
}


In [25]:
import ee

def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    results = {}
    
    for prop, collection_id in properties.items():
        try:
            image = ee.Image(collection_id)
            band_name = f"{prop}_0-5cm_mean"
            
            # Usamos Reducer.mean() sobre la región completa
            val = image.reduceRegion(
                reducer=ee.Reducer.mean(), 
                geometry=region,
                scale=250, # Resolución nativa de SoilGrids
                bestEffort=True
            ).get(band_name)
            
            results[prop] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
soil_data = get_soil_profile_area(7.3297, -73.1867)
print("Perfil de suelo promedio (1ha):", soil_data)

print(json.dumps(soil_data, indent=4))

Perfil de suelo promedio (1ha): {'phh2o': 52.509803921568626, 'soc': 568.9019607843137, 'clay': 344.4705882352941, 'sand': 312.01960784313724, 'silt': 344.2549019607843, 'bdod': 99.25490196078431, 'cec': 186.90196078431373}
{
    "phh2o": 52.509803921568626,
    "soc": 568.9019607843137,
    "clay": 344.4705882352941,
    "sand": 312.01960784313724,
    "silt": 344.2549019607843,
    "bdod": 99.25490196078431,
    "cec": 186.90196078431373
}


In [ ]:
import ee
ee.Initialize()

In [27]:
def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    results = {}
    
    for prop, collection_id in properties.items():
        try:
            image = ee.Image(collection_id)
            band_name = f"{prop}_5-15cm_mean"
            
            # Usamos Reducer.mean() sobre la región completa
            val = image.reduceRegion(
                reducer=ee.Reducer.mean(), 
                geometry=region,
                scale=250, # Resolución nativa de SoilGrids
                bestEffort=True
            ).get(band_name)
            
            results[prop] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
soil_data = get_soil_profile_area(7.3297, -73.1867)
print("Perfil de suelo promedio (1ha):", soil_data)

print(json.dumps(soil_data, indent=4))

Perfil de suelo promedio (1ha): {'phh2o': 53.25490196078431, 'soc': 387.764705882353, 'clay': 351.9411764705882, 'sand': 314.235294117647, 'silt': 334.5686274509804, 'bdod': 102.50980392156862, 'cec': 184.80392156862743}
{
    "phh2o": 53.25490196078431,
    "soc": 387.764705882353,
    "clay": 351.9411764705882,
    "sand": 314.235294117647,
    "silt": 334.5686274509804,
    "bdod": 102.50980392156862,
    "cec": 184.80392156862743
}


In [ ]:
def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    results = {}
    
    for prop, collection_id in properties.items():
        try:
            image = ee.Image(collection_id)
            band_name = f"{prop}_15-30cm_mean"
            
            # Usamos Reducer.mean() sobre la región completa
            val = image.reduceRegion(
                reducer=ee.Reducer.mean(), 
                geometry=region,
                scale=250, # Resolución nativa de SoilGrids
                bestEffort=True
            ).get(band_name)
            
            results[prop] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
soil_data = get_soil_profile_area(7.3297, -73.1867)
print("Perfil de suelo promedio (1ha):", soil_data)

print(json.dumps(soil_data, indent=4))

Perfil de suelo promedio (1ha): {'phh2o': 53, 'soc': 222.7058823529412, 'clay': 375.2156862745098, 'sand': 311.4705882352941, 'silt': 313.5686274509804, 'bdod': 107.76470588235294, 'cec': 168.2941176470588}
{
    "phh2o": 53,
    "soc": 222.7058823529412,
    "clay": 375.2156862745098,
    "sand": 311.4705882352941,
    "silt": 313.5686274509804,
    "bdod": 107.76470588235294,
    "cec": 168.2941176470588
}


In [29]:
def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    results = {}
    
    for prop, collection_id in properties.items():
        try:
            image = ee.Image(collection_id)
            band_name = f"{prop}_30-60cm_mean"
            
            # Usamos Reducer.mean() sobre la región completa
            val = image.reduceRegion(
                reducer=ee.Reducer.mean(), 
                geometry=region,
                scale=250, # Resolución nativa de SoilGrids
                bestEffort=True
            ).get(band_name)
            
            results[prop] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
soil_data = get_soil_profile_area(7.3297, -73.1867)
print("Perfil de suelo promedio (1ha):", soil_data)

print(json.dumps(soil_data, indent=4))

Perfil de suelo promedio (1ha): {'phh2o': 53.25490196078431, 'soc': 161.5686274509804, 'clay': 427.68627450980387, 'sand': 280.96078431372547, 'silt': 291.09803921568624, 'bdod': 112.50980392156863, 'cec': 162.78431372549016}
{
    "phh2o": 53.25490196078431,
    "soc": 161.5686274509804,
    "clay": 427.68627450980387,
    "sand": 280.96078431372547,
    "silt": 291.09803921568624,
    "bdod": 112.50980392156863,
    "cec": 162.78431372549016
}


In [ ]:
import ee
import json

def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm']
    
    results = {}
    
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                
                # Usamos Reducer.mean() sobre la región completa para cada profundidad
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(), 
                    geometry=region,
                    scale=250, # Resolución nativa de SoilGrids
                    bestEffort=True
                ).get(band_name)
                
                results[prop][depth] = val.getInfo()
                
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
# Finca Matanza 7.300921,-73.009794
soil_data = get_soil_profile_area(3.580109040361371, -76.31299479308868)
print("Perfil de suelo completo por profundidades (1ha):")
print(json.dumps(soil_data, indent=4))

Perfil de suelo completo por profundidades (1ha):
{
    "phh2o": {
        "0-5cm": 57,
        "5-15cm": 56.5,
        "15-30cm": 56,
        "30-60cm": 57
    },
    "soc": {
        "0-5cm": 633.1800000000001,
        "5-15cm": 389.29999999999995,
        "15-30cm": 279.54,
        "30-60cm": 184.04
    },
    "clay": {
        "0-5cm": 352.00000000000006,
        "5-15cm": 354.49999999999994,
        "15-30cm": 392.50000000000006,
        "30-60cm": 444.5
    },
    "sand": {
        "0-5cm": 309,
        "5-15cm": 303.5,
        "15-30cm": 287.5,
        "30-60cm": 268.5
    },
    "silt": {
        "0-5cm": 339,
        "5-15cm": 342,
        "15-30cm": 320,
        "30-60cm": 287
    },
    "bdod": {
        "0-5cm": 100.32000000000001,
        "5-15cm": 103.92,
        "15-30cm": 108.89999999999999,
        "30-60cm": 111.7
    },
    "cec": {
        "0-5cm": 209.00000000000003,
        "5-15cm": 181.50000000000003,
        "15-30cm": 173.49999999999997,
        "30-60cm": 167

In [10]:
import pandas as pd

# Assuming 'soil_data' is your nested dictionary result
df = pd.DataFrame(soil_data).T
df.index.name = 'property'
df.reset_index(inplace=True)
print(df.head(10))

  property   0-5cm  5-15cm  15-30cm  30-60cm
0    phh2o   57.00   56.50    56.00    57.00
1      soc  633.18  389.30   279.54   184.04
2     clay  352.00  354.50   392.50   444.50
3     sand  309.00  303.50   287.50   268.50
4     silt  339.00  342.00   320.00   287.00
5     bdod  100.32  103.92   108.90   111.70
6      cec  209.00  181.50   173.50   167.50


In [7]:
import pandas as pd

# Assuming 'soil_data' is your nested dictionary result
df = pd.DataFrame(soil_data).T
df.index.name = 'property'
df.reset_index(inplace=True)

# Save to CSV
# df.to_csv("soil_profile_data.csv", index=False)
# print("CSV export successful!")

In [11]:
sg_df = pd.read_csv("soil_profile_data.csv")
print(sg_df.head(10))
# Finca Matanza 7.300921,-73.009794

  property       0-5cm      5-15cm     15-30cm     30-60cm
0    phh2o   52.509804   53.254902   53.000000   53.254902
1      soc  568.901961  387.764706  222.705882  161.568627
2     clay  344.470588  351.941176  375.215686  427.686275
3     sand  312.019608  314.235294  311.470588  280.960784
4     silt  344.254902  334.568627  313.568627  291.098039
5     bdod   99.254902  102.509804  107.764706  112.509804
6      cec  186.901961  184.803922  168.294118  162.784314
